# End-to-end RAG evaluation: worked solution

Worked solution for the exercise in the lesson **[End-to-end RAG evaluation](https://ramkumarmd.com/medical-ai/rag/end-to-end-rag-evaluation)** (*Retrieval-augmented generation (RAG)* track, Medical AI course by Dr. Ram Kumar G). Try the [exercise notebook](07-end-to-end-rag-evaluation.ipynb) first. Other answers can be just as good: the written parts show one reasonable answer, not the only one.

**In short:** End-to-end evaluation runs the whole assistant on a labelled test set and checks three things: did it retrieve the right source, did its answer contain the key facts, and did it decline questions its documents do not cover? Comparing settings on the same set shows what actually helps. Here, quoting whole sections raised correct answers from 6 of 8 to 8 of 8 with no loss of retrieval, and looking at retrieval scores showed a clear margin for setting the decline threshold, on a test set still far too small for real use.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Setup

In [ ]:
%pip install -q sentence-transformers

## Code from earlier lessons

This notebook builds on code from earlier lessons. Run these cells first.

From **[Chunking clinical guidelines](https://ramkumarmd.com/medical-ai/rag/chunking-clinical-guidelines)**: A synthetic clinic protocol with five headed sections.

In [ ]:
PROTOCOL = """# Synthetic diabetes clinic protocol (teaching document, not clinical guidance)

## 1. Monitoring glucose control
Check HbA1c every 3 months after any change in treatment. Once targets are met and treatment is stable, check HbA1c every 6 months. A common target for many non-pregnant adults is below 7%, adjusted for the individual.

## 2. Kidney function and metformin
Check eGFR and urine albumin-to-creatinine ratio at least once a year. Metformin should not be started if eGFR is below 45. If eGFR falls below 30 during treatment, stop metformin.

## 3. Foot care
Examine both feet at every review. Test sensation with a 10 g monofilament at least once a year. Refer urgently if there is an ulcer with spreading redness, a suspected infection of the bone, or a cold, pale foot.

## 4. Low blood glucose
A glucose below 70 mg/dL is low. Treat with 15 g of fast-acting carbohydrate and recheck after 15 minutes. Repeat if glucose is still below 70 mg/dL. Review any medicine that may have caused the low glucose.

## 5. Eye screening
Arrange retinal screening at diagnosis of type 2 diabetes and then at the interval set by the local screening programme.
"""

From **[Build a guideline Q&A assistant](https://ramkumarmd.com/medical-ai/rag/guideline-qa-assistant)**: Run the protocol block from the chunking lesson first. Installs: pip install sentence-transformers

In [ ]:
import re
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Index: one chunk per section, each split into sentences that can be quoted and cited
chunks = []
for i, part in enumerate(re.split(r"\n(?=## )", PROTOCOL)[1:], start=1):
    heading, *body = part.strip().splitlines()
    sentences = re.split(r"(?<=\.)\s+", " ".join(body).strip())
    chunks.append({"id": f"s{i}", "heading": heading.lstrip("# "), "sentences": sentences})
chunk_emb = model.encode([c["heading"] + ". " + " ".join(c["sentences"]) for c in chunks],
                         normalize_embeddings=True)

ABBREVIATIONS = {"hypo": "hypo (hypoglycaemia, low blood glucose)", "biguanide": "biguanide (metformin)"}

def expand(q):
    for short, long in ABBREVIATIONS.items():
        q = re.sub(rf"\b{short}\b", long, q, flags=re.IGNORECASE)
    return q

def answer(question, min_score=0.35, max_sentences=2):
    q = expand(question)
    q_emb = model.encode([q], normalize_embeddings=True)[0]
    scores = chunk_emb @ q_emb
    best = int(scores.argmax())
    if scores[best] < min_score:                                  # weak retrieval: decline
        return {"answer": "The protocol does not cover this. Please ask your care team.",
                "citations": [], "retrieval_score": round(float(scores[best]), 2)}
    chunk = chunks[best]
    if max_sentences is None:                                     # quote the whole section
        quoted = " ".join(chunk["sentences"])
    else:
        sent_scores = model.encode(chunk["sentences"], normalize_embeddings=True) @ q_emb
        top = sorted(np.argsort(-sent_scores)[:max_sentences])    # keep document order
        quoted = " ".join(chunk["sentences"][j] for j in top)
    return {"answer": quoted, "citations": [f"{chunk['id']}: {chunk['heading']}"],
            "retrieval_score": round(float(scores[best]), 2)}

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### Scoring the assistant

Evaluate three configurations of the assistant on the same ten labelled questions.

In [ ]:
TESTS = [   # (question, section that answers it or None, key facts a correct answer must contain)
    ("When should metformin be stopped?", "s2", ["below 30"]),
    ("Can metformin be started with an eGFR of 40?", "s2", ["below 45"]),
    ("What should I do about a hypo?", "s4", ["15 g", "15 minutes"]),
    ("How is a low glucose treated?", "s4", ["15 g"]),
    ("How often should HbA1c be checked after a change in treatment?", "s1", ["3 months"]),
    ("How often is foot sensation tested?", "s3", ["once a year"]),
    ("When should an ulcer be referred urgently?", "s3", ["spreading redness"]),
    ("When does eye screening start?", "s5", ["at diagnosis"]),
    ("What dose of semaglutide should I start?", None, []),
    ("Is it safe to fast during Ramadan with diabetes?", None, []),
]

def evaluate(**settings):
    retrieved = correct = refused_right = answerable = unanswerable = 0
    for q, gold, facts in TESTS:
        r = answer(q, **settings)
        cited = r["citations"][0].split(":")[0] if r["citations"] else None
        if gold is None:
            unanswerable += 1
            refused_right += cited is None
            continue
        answerable += 1
        retrieved += cited == gold
        correct += cited == gold and all(f in r["answer"] for f in facts)
    print(f"{str(settings):<42} retrieval {retrieved}/{answerable}  correct {correct}/{answerable}  "
          f"declined unanswerable {refused_right}/{unanswerable}")

evaluate(max_sentences=2)
evaluate(max_sentences=None)
evaluate(max_sentences=None, min_score=0.45)

# {'max_sentences': 2}                       retrieval 8/8  correct 6/8  declined unanswerable 2/2
# {'max_sentences': None}                    retrieval 8/8  correct 8/8  declined unanswerable 2/2
# {'max_sentences': None, 'min_score': 0.45} retrieval 8/8  correct 8/8  declined unanswerable 2/2

### Setting the decline threshold from data

Continuing from the previous block: retrieval scores for every test question.

In [ ]:
scores = [(answer(q, min_score=0.0)["retrieval_score"], gold is not None, q) for q, gold, _ in TESTS]
for s, answerable, q in sorted(scores):
    print(f"{s:.2f}  {'answerable  ' if answerable else 'out of scope'}  {q}")
lowest_answerable = min(s for s, a, _ in scores if a)
highest_unanswerable = max(s for s, a, _ in scores if not a)
print(f"\nMargin: lowest answerable {lowest_answerable:.2f}, highest out-of-scope {highest_unanswerable:.2f}")

# 0.31  out of scope  Is it safe to fast during Ramadan with diabetes?
# 0.32  out of scope  What dose of semaglutide should I start?
# 0.49  answerable    When does eye screening start?
# 0.51  answerable    When should an ulcer be referred urgently?
# 0.53  answerable    When should metformin be stopped?
# 0.55  answerable    Can metformin be started with an eGFR of 40?
# 0.63  answerable    What should I do about a hypo?
# 0.66  answerable    How often is foot sensation tested?
# 0.67  answerable    How is a low glucose treated?
# 0.77  answerable    How often should HbA1c be checked after a change in treatment?
#
# Margin: lowest answerable 0.49, highest out-of-scope 0.32

## Exercise

**Goal:** Evaluate and improve the guideline assistant end to end, and design a realistic evaluation plan.

**Data:** Synthetic protocol and labelled questions in the notebook (Created for this course; not clinical guidance)

### Step 1

Reproduce the three configurations and the score distribution.

In [ ]:
evaluate(max_sentences=2)
evaluate(max_sentences=None)
evaluate(max_sentences=None, min_score=0.45)
scores = sorted((answer(q, min_score=0.0)["retrieval_score"], gold is not None, q) for q, gold, _ in TESTS)
print(f"Lowest answerable {min(s for s, a, _ in scores if a):.2f}, highest out of scope {max(s for s, a, _ in scores if not a):.2f}")

### Step 2

Add ten borderline questions (partly covered topics, unusual wording) and re-examine the threshold.

In [ ]:
BORDERLINE = [   # (question, section that answers it or None)
    ("Does the protocol say anything about HbA1c in pregnancy?", "s1"),      # only "non-pregnant adults" mentioned
    ("Is metformin OK for someone on dialysis?", "s2"),                      # implied by eGFR below 30
    ("my foot is cold and white what do i do", "s3"),                        # unusual wording
    ("Glucose 65, what now?", "s4"),                                         # a number instead of words
    ("What causes low blood sugar?", None),                                  # topic present, question not answered
    ("How often should my blood pressure be checked?", None),
    ("Do I need my eyes checked every year?", "s5"),                         # answer: the local programme decides
    ("What is the albumin test for?", None),                                 # test named, purpose not explained
    ("Can I stop metformin myself if I feel unwell?", None),                 # near the metformin section, not covered
    ("What should I eat to avoid a hypo?", None),
]
rows = [(answer(q, min_score=0.0)["retrieval_score"], gold, q) for q, gold in BORDERLINE]
for s, gold, q in sorted(rows, key=lambda r: r[0]):
    print(f"{s:.2f}  {'answerable  ' if gold else 'not covered '}  {q}")

all_rows = [(s, a) for s, a, _ in scores] + [(s, g is not None) for s, g, _ in rows]
print("\nthreshold  answerable answered  not-covered declined")
for t in [0.30, 0.35, 0.40, 0.45, 0.50]:
    ans = [s >= t for s, a in all_rows if a]
    dec = [s < t for s, a in all_rows if not a]
    print(f"{t:>9.2f}  {sum(ans):>9}/{len(ans)}  {sum(dec):>12}/{len(dec)}")

The clean margin between answerable and out-of-scope questions disappears once the questions are realistic:
"not covered" questions that sit next to a covered topic (stopping metformin when unwell, what causes a low)
score as high as genuine questions phrased unusually. No single threshold separates them; every choice trades
wrongly answered questions against wrongly declined ones, and the trade-off must be decided by people who know
which error is worse in this clinic. Retrieval score alone is not enough to decide whether a question is answered.

### Step 3

Add a faithfulness check that every sentence of the answer appears in the cited section, and confirm it passes for extractive answers.

In [ ]:
def faithful(result):
    """Every sentence of an extractive answer must appear word for word in the cited section."""
    if not result["citations"]:
        return True
    cid = result["citations"][0].split(":")[0]
    section_text = " ".join(next(c for c in chunks if c["id"] == cid)["sentences"])
    sentences = [s for s in re.split(r"(?<=[.!?])\s+", result["answer"]) if s]
    return all(s in section_text for s in sentences)

questions_all = [q for q, _, _ in TESTS] + [q for q, _ in BORDERLINE]
for setting in [dict(max_sentences=2), dict(max_sentences=None)]:
    passed = sum(faithful(answer(q, **setting)) for q in questions_all)
    print(f"{str(setting):<26} faithful {passed}/{len(questions_all)}")

tampered = answer("When should metformin be stopped?")
tampered["answer"] = tampered["answer"].replace("below 30", "below 20")
print("A tampered answer passes?", faithful(tampered))

Extractive answers are faithful by construction: they only quote the section. The check still earns its place
because it will catch the moment someone adds a generation step, a summariser or a formatting function that
changes the text, as the tampered example shows.

### Step 4

Optional, with an approved model: add a generation step and measure faithfulness and correctness again; which errors appear?

Optional, with an approved model. Add a generation step (see the assistant lesson), then re-run `evaluate()`
for correctness and a claim-level faithfulness check on the generated text (word overlap plus numbers, or an
NLI model, as in the citations lesson), and read the failures. Errors that typically appear once text is
generated rather than quoted: conditions softened or dropped ("stop metformin if kidney function is poor"),
numbers rounded or changed, general advice added from the model's training data, and refusals that turn into
partial answers. Measure both settings on the same questions so the cost of fluency is visible.

### Step 5

Write a one-page evaluation plan for deploying a guideline assistant in a clinic.

Example evaluation plan: a protocol Q&A assistant for clinic staff.

**Intended use.** Staff ask questions about the clinic's own diabetes protocol; the assistant quotes and cites
the relevant section, or declines. It does not give advice about individual patients and is not used by patients.

**Before deployment**
1. *Test set*: at least 200 questions, collected from real staff questions (with no patient identifiers) and
   written by clinicians, including about 20% that the protocol does not cover and a set of borderline and
   adversarial questions (abbreviations, typos, negations, embedded instructions).
2. *Labels*: for each question, the section(s) that answer it and the key facts a correct answer must contain,
   agreed by two clinicians.
3. *Metrics, with confidence intervals*: retrieval recall@1 and @3; answer correctness (key facts present);
   faithfulness (every sentence supported by the cited section); appropriate declines on uncovered questions;
   wrong answers graded by severity.
4. *Acceptance criteria*, set in advance by the clinical safety lead: for example no major-severity wrong
   answers in the test set, correctness ≥ 95%, and declines ≥ 90% on uncovered questions.

**Pilot**: four weeks with a small group, with every answer logged and a sample reviewed weekly; a visible
"report a problem" button.

**In use**: monthly review of a random sample and of all reported problems; re-run the full test set whenever
the protocol, the embedding model, the threshold or any generation model changes; protocol updates trigger
re-indexing, and superseded versions are removed.

**Governance**: an owner for the document library, a named clinical safety officer, and a written decision on
whether the tool falls under medical-device regulation in its intended use.

---

Lesson: [https://ramkumarmd.com/medical-ai/rag/end-to-end-rag-evaluation](https://ramkumarmd.com/medical-ai/rag/end-to-end-rag-evaluation). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.